# 22 · Temporal proximity audit — ảnh cùng bệnh nhân chụp cách nhau ≤ 36 h

Đếm trên **cohort đã select** (`cohort_v2`, 211,130 study). Timestamp là `index_time` ở mức
**study** (MIMIC-CXR chỉ có `StudyDate`/`StudyTime`, không có giờ riêng cho từng ảnh), nên:

* hai ảnh **trong cùng một study** ⇒ gap = 0 giờ theo định nghĩa — đếm tách riêng;
* hai ảnh **khác study, cùng subject** ⇒ gap = `|index_time_i − index_time_j|`.

CPU thuần, ~10–20 giây, không cần GPU và không cần stage ảnh.

## C0 · Load cohort + timestamp + số ảnh mỗi study

In [ ]:
import os, json, math
import numpy as np, pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = '.'
OUT_DIR = os.path.join(DATA_DIR, 'cohort_v2')
AUD_DIR = os.path.join(DATA_DIR, 'temporal_audit'); os.makedirs(AUD_DIR, exist_ok=True)

WIN_H     = 36.0                                  # <<< cua so chinh
BINS_H    = [0, 1, 6, 12, 24, 36]                 # bin de bao cao chi tiet
EXTRA_WIN = [6, 12, 24, 36, 48, 72, 168]          # doi chieu nhieu nguong

ehr = (pd.read_parquet(os.path.join(OUT_DIR, 'cohort_study_level.parquet'),
                       columns=['study_id', 'subject_id', 'split', 'setting'])
       .drop_duplicates('study_id'))
_bcols = pd.read_parquet(os.path.join(OUT_DIR, 'cohort_base.parquet')).columns.tolist()
_want = ['study_id', 'index_time'] + [c for c in ('cur_hadm', 'cur_edstay') if c in _bcols]
base = (pd.read_parquet(os.path.join(OUT_DIR, 'cohort_base.parquet'), columns=_want)
        .drop_duplicates('study_id'))

vm = pd.read_parquet(os.path.join(OUT_DIR, 'view_map.parquet'))
vm = vm[vm.view_group.isin(['frontal', 'lateral'])]
img = (vm.groupby('study_id')
       .agg(n_img=('dicom_id', 'size'),
            vsig=('view_group', lambda s: ''.join(sorted({x[0].upper() for x in s}))))
       .reset_index())

D = ehr.merge(base, on='study_id', how='left').merge(img, on='study_id', how='left')
D['n_img'] = D.n_img.fillna(0).astype('int64')
D['vsig'] = D.vsig.fillna('')
D['t'] = pd.to_datetime(D.index_time)
print(f'cohort {len(D):,} study | {D.subject_id.nunique():,} subject | anh {int(D.n_img.sum()):,}')
print(f'  split: {dict(D.split.value_counts())}')
print(f'  missing index_time: {int(D.t.isna().sum()):,} | study khong co anh: {int((D.n_img==0).sum()):,}')

D = D[D.t.notna() & (D.n_img > 0)].copy()
D['tsec'] = D.t.astype('int64') / 1e9
D = D.sort_values(['subject_id', 'tsec', 'study_id']).reset_index(drop=True)
print(f'  -> dung {len(D):,} study co ca timestamp lan anh')

## C1 · Đếm cặp study cùng subject có gap ≤ 36 h

Với mỗi subject, sort theo thời gian rồi `searchsorted` — đếm cặp `O(n log n)`, không dựng ma trận
`n×n`. `n_img_pairs` = `n_img_i × n_img_j`, tức số **cặp ảnh** sinh ra từ cặp study đó.

In [ ]:
sub  = D.subject_id.to_numpy()
tsec = D.tsec.to_numpy()
bnd  = np.r_[0, np.flatnonzero(np.diff(sub)) + 1, len(sub)]
print(f'{len(bnd)-1:,} subject | study/subject: median '
      f'{np.median(np.diff(bnd)):.0f}, max {np.diff(bnd).max():,}')

def pairs_within(win_h):
    '''Tra ve (i, j) — chi so toan cuc trong D, i<j, cung subject, gap <= win_h.'''
    W = win_h * 3600.0
    I, J = [], []
    for a, b in zip(bnd[:-1], bnd[1:]):
        t = tsec[a:b]
        if len(t) < 2: continue
        hi = np.searchsorted(t, t + W, side='right')
        cnt = hi - np.arange(len(t)) - 1
        if cnt.sum() == 0: continue
        i = np.repeat(np.arange(len(t)), cnt)
        off = np.arange(cnt.sum()) - np.repeat(np.cumsum(cnt) - cnt, cnt)
        j = i + 1 + off
        I.append(i + a); J.append(j + a)
    if not I: return np.array([], 'int64'), np.array([], 'int64')
    return np.concatenate(I), np.concatenate(J)

I36, J36 = pairs_within(WIN_H)
gap_h = (tsec[J36] - tsec[I36]) / 3600.0
n_img = D.n_img.to_numpy()
img_pairs = n_img[I36].astype('int64') * n_img[J36].astype('int64')

print(f'\n=== CUA SO {WIN_H:.0f} GIO ===')
print(f'  cap STUDY   cung subject, gap <= {WIN_H:.0f}h : {len(I36):>10,}')
print(f'  cap ANH     sinh ra tu cac cap study do   : {int(img_pairs.sum()):>10,}')
print(f'  study co >=1 hang xom <= {WIN_H:.0f}h          : '
      f'{len(np.unique(np.r_[I36, J36])):>10,} / {len(D):,} '
      f'({len(np.unique(np.r_[I36, J36]))/len(D):.1%})')
_im = int(n_img[np.unique(np.r_[I36, J36])].sum())
print(f'  ANH thuoc cac study do                    : {_im:>10,} / {int(n_img.sum()):,} '
      f'({_im/n_img.sum():.1%})')
print(f'  subject co >=1 cap nhu vay                : '
      f'{len(np.unique(sub[I36])):>10,} / {len(bnd)-1:,}')

## C2 · Phân rã theo khoảng gap · split · view · encounter

In [ ]:
lab = [f'{BINS_H[k]}-{BINS_H[k+1]}h' for k in range(len(BINS_H)-1)]
gbin = pd.cut(gap_h, BINS_H, labels=lab, include_lowest=True)
splt = D.split.to_numpy(); vsig = D.vsig.to_numpy()
hadm = D.cur_hadm.to_numpy() if 'cur_hadm' in D else np.full(len(D), np.nan)

P = pd.DataFrame(dict(
    subject_id=sub[I36], study_i=D.study_id.to_numpy()[I36], study_j=D.study_id.to_numpy()[J36],
    gap_h=gap_h, gap_bin=gbin, img_pairs=img_pairs,
    n_img_i=n_img[I36], n_img_j=n_img[J36],
    split_i=splt[I36], split_j=splt[J36], same_split=splt[I36] == splt[J36],
    vsig_i=vsig[I36], vsig_j=vsig[J36], same_vsig=vsig[I36] == vsig[J36],
    share_view=[bool(set(a) & set(b)) for a, b in zip(vsig[I36], vsig[J36])],
    same_hadm=(pd.notna(hadm[I36]) & (hadm[I36] == hadm[J36]))))
P.to_parquet(f'{AUD_DIR}/pairs_within_{int(WIN_H)}h.parquet', index=False)

print(f'=== THEO KHOANG GAP (<= {WIN_H:.0f}h) ===')
B = (P.groupby('gap_bin', observed=False)
     .agg(n_pair_study=('gap_h', 'size'), n_pair_image=('img_pairs', 'sum'),
          n_subject=('subject_id', 'nunique'), same_hadm=('same_hadm', 'sum'),
          share_view=('share_view', 'sum')))
B['pct_pair'] = B.n_pair_study / max(len(P), 1)
print(B.to_string(float_format=lambda v: f'{v:.4f}'))
print(f'\n  gap == 0 dung bang 0.0h (cung timestamp, khac study): '
      f'{int((P.gap_h == 0).sum()):,} cap')

print(f'\n=== THEO SPLIT ===')
S = (P.groupby(['split_i', 'split_j'], observed=True)
     .agg(n_pair=('gap_h', 'size'), n_pair_image=('img_pairs', 'sum')))
print(S.to_string())
print(f'  cap VUOT split (leak neighbour): {int((~P.same_split).sum()):,} '
      f'({(~P.same_split).mean():.2%}) — split theo subject nen ky vong = 0')

print(f'\n=== THEO VIEW ===')
print(f'  cung view signature : {int(P.same_vsig.sum()):>9,} ({P.same_vsig.mean():.1%})')
print(f'  co it nhat 1 view chung: {int(P.share_view.sum()):>9,} ({P.share_view.mean():.1%})')
print(P.groupby(['vsig_i', 'vsig_j'], observed=True).size().sort_values(ascending=False)
      .head(9).to_string())
if pd.notna(hadm).any():
    print(f'\n=== ENCOUNTER ===')
    print(f'  cung cur_hadm: {int(P.same_hadm.sum()):,} ({P.same_hadm.mean():.1%}) '
          f'| khac/thieu hadm: {int((~P.same_hadm).sum()):,}')

## C3 · Đối chiếu nhiều ngưỡng · nearest-neighbour · cụm (burst)

*Cụm* = thành phần liên thông khi nối hai study cùng subject có gap ≤ 36 h (nối chuỗi, nên một cụm
có thể trải dài hơn 36 h). Cho biết ảnh bị "dồn cục" tới mức nào.

In [ ]:
print(f'=== SO SANH NGUONG ({len(D):,} study, {int(n_img.sum()):,} anh) ===')
rows = []
for w in EXTRA_WIN:
    i_, j_ = pairs_within(w)
    st = np.unique(np.r_[i_, j_])
    rows.append(dict(window_h=w, n_pair_study=len(i_),
                     n_pair_image=int((n_img[i_].astype('int64')*n_img[j_]).sum()) if len(i_) else 0,
                     n_study_with_neighbour=len(st), pct_study=len(st)/len(D),
                     n_image_involved=int(n_img[st].sum()) if len(st) else 0,
                     pct_image=(n_img[st].sum()/n_img.sum()) if len(st) else 0.0,
                     n_subject=len(np.unique(sub[i_])) if len(i_) else 0))
W = pd.DataFrame(rows); W.to_csv(f'{AUD_DIR}/window_sweep.csv', index=False)
print(W.to_string(index=False, float_format=lambda v: f'{v:.4f}'))

# ---------------- nearest neighbour cua tung study ----------------
nn = np.full(len(D), np.inf)
for a, b in zip(bnd[:-1], bnd[1:]):
    t = tsec[a:b]
    if len(t) < 2: continue
    d = np.diff(t)
    nn[a:b-1] = np.minimum(nn[a:b-1], d)
    nn[a+1:b] = np.minimum(nn[a+1:b], d)
nn_h = nn/3600.0
D['nn_gap_h'] = nn_h
D[['study_id', 'subject_id', 'split', 'n_img', 'vsig', 'nn_gap_h']].to_parquet(
    f'{AUD_DIR}/study_nearest_neighbour.parquet', index=False)
_f = np.isfinite(nn_h)
print(f'\n=== NEAREST NEIGHBOUR (study gan nhat cung subject) ===')
print(f'  study co it nhat 1 study khac cung subject: {int(_f.sum()):,} / {len(D):,}')
print('  percentile gap (gio): ' + ' · '.join(
    f'p{q}={np.percentile(nn_h[_f], q):.1f}' for q in (1, 5, 10, 25, 50, 75, 90)))
for w in (6, 12, 24, 36):
    print(f'  <= {w:>3}h : {int((nn_h <= w).sum()):>9,} study ({(nn_h <= w).mean():.1%}) | '
          f'{int(n_img[nn_h <= w].sum()):>9,} anh ({n_img[nn_h <= w].sum()/n_img.sum():.1%})')

# ---------------- cum (burst) theo chuoi <= WIN_H ----------------
W_S = WIN_H*3600.0
cid = np.zeros(len(D), 'int64'); c = 0
for a, b in zip(bnd[:-1], bnd[1:]):
    t = tsec[a:b]
    brk = np.r_[True, np.diff(t) > W_S]
    cid[a:b] = c + np.cumsum(brk) - 1
    c = cid[a:b].max() + 1
D['cluster'] = cid
sz = D.groupby('cluster').agg(n_study=('study_id', 'size'), n_img=('n_img', 'sum'))
print(f'\n=== CUM (chuoi lien tiep, moi buoc <= {WIN_H:.0f}h) ===')
print(f'  {len(sz):,} cum | cum >=2 study: {int((sz.n_study >= 2).sum()):,}')
vc = sz.n_study.value_counts().sort_index()
print('  phan bo so study/cum:')
for k, v in vc.head(8).items():
    _im = int(sz.n_img[sz.n_study == k].sum())
    print(f'    {k:>3} study : {v:>8,} cum | {_im:>9,} anh ({_im/n_img.sum():.1%})')
if len(vc) > 8:
    _big = sz[sz.n_study > vc.index[7]]
    print(f'    >{vc.index[7]:>2} study : {len(_big):>8,} cum | {int(_big.n_img.sum()):>9,} anh '
          f'| max {int(sz.n_study.max())} study/cum')
sz.to_csv(f'{AUD_DIR}/cluster_sizes.csv')
print(f'\nartifacts -> {AUD_DIR}/')

## C4 · Study "gốc" có **bao nhiêu** hàng xóm ≤ 36 h — và bao nhiêu ảnh quanh nó

`degree` của một study = số study **khác** của cùng subject nằm trong ±36 h (đối xứng, tính cả trước
lẫn sau). `n_img_36h` = số ảnh trong bán kính 36 h quanh study đó, **kể cả ảnh của chính nó**.

Câu hỏi: có study nào là "gốc" của cả một chùm nhiều study ≤ 36 h, hay chỉ toàn cặp đôi lẻ?

In [ ]:
n = len(D)
deg = (np.bincount(I36, minlength=n) + np.bincount(J36, minlength=n)).astype('int64')
img_nb = n_img.astype('int64').copy()                 # bat dau bang anh cua chinh no
np.add.at(img_nb, I36, n_img[J36].astype('int64'))
np.add.at(img_nb, J36, n_img[I36].astype('int64'))
D['deg_36h'] = deg; D['n_img_36h'] = img_nb
has = deg > 0

def _st(name, v):
    v = np.asarray(v)
    if not len(v): return dict(nhom=name, n=0)
    return dict(nhom=name, n=len(v), mean=float(v.mean()), std=float(v.std()),
                p50=float(np.percentile(v, 50)), p90=float(np.percentile(v, 90)),
                p99=float(np.percentile(v, 99)), max=int(v.max()))

print(f'=== SO ANH MOI STUDY (n_img) ===')
print(pd.DataFrame([_st('tat ca study', n_img)]).to_string(index=False, float_format=lambda v: f'{v:.3f}'))

print(f'\n=== DEGREE — so study KHAC cung subject trong +/-{WIN_H:.0f}h ===')
print(pd.DataFrame([_st('tat ca study', deg), _st(f'chi study co >=1 hang xom', deg[has])])
      .to_string(index=False, float_format=lambda v: f'{v:.3f}'))

print(f'\n=== SO ANH TRONG BAN KINH {WIN_H:.0f}h (ke ca anh cua chinh study) ===')
print(pd.DataFrame([_st('tat ca study', img_nb), _st('chi study co >=1 hang xom', img_nb[has])])
      .to_string(index=False, float_format=lambda v: f'{v:.3f}'))

print(f'\n=== PHAN BO DEGREE ===')
cuts = [-1, 0, 1, 2, 5, 10, 20, 10**9]
labs = ['0 (le loi)', '1', '2', '3-5', '6-10', '11-20', '>20']
db = pd.cut(deg, cuts, labels=labs)
T = pd.DataFrame(dict(deg_bin=db, n_img=n_img, n_img_36h=img_nb)).groupby('deg_bin', observed=False).agg(
    n_study=('n_img', 'size'), anh_cua_study=('n_img', 'sum'),
    mean_img_36h=('n_img_36h', 'mean'), max_img_36h=('n_img_36h', 'max'))
T['pct_study'] = T.n_study / n
print(T.to_string(float_format=lambda v: f'{v:.3f}'))
_multi = int((deg >= 2).sum())
print(f'\n  study co >=2 hang xom <= {WIN_H:.0f}h : {_multi:,} ({_multi/n:.2%}) '
      f'-> {"CO chum, khong chi la cap doi" if _multi else "chi toan cap doi"}')
print(f'  degree lon nhat: {int(deg.max())} | so anh quanh no lon nhat: {int(img_nb.max()):,}')

print(f'\n=== TOP 15 STUDY "GOC" (degree cao nhat) ===')
top = D.loc[np.argsort(-deg, kind='stable')[:15],
            ['study_id', 'subject_id', 'split', 't', 'vsig', 'n_img', 'deg_36h', 'n_img_36h']]
print(top.to_string(index=False))

print(f'\n=== SUBJECT dong dac nhat (theo degree lon nhat cua subject do) ===')
sd = (D.groupby('subject_id').agg(n_study=('study_id', 'size'), max_deg=('deg_36h', 'max'),
                                  tong_anh=('n_img', 'sum'), max_img_36h=('n_img_36h', 'max'))
      .sort_values('max_deg', ascending=False))
print(sd.head(10).to_string())
print(f'  subject co it nhat 1 study degree>=2: {int((sd.max_deg >= 2).sum()):,} / {len(sd):,}')

D[['study_id', 'subject_id', 'split', 'n_img', 'vsig', 'nn_gap_h', 'deg_36h', 'n_img_36h']].to_parquet(
    f'{AUD_DIR}/study_degree_{int(WIN_H)}h.parquet', index=False)
print(f'\n-> {AUD_DIR}/study_degree_{int(WIN_H)}h.parquet')